# 02. 데이터셋 & YAML — 커스텀 데이터 준비

학습을 하려면 YOLO가 이해하는 **데이터셋 형식**과 **YAML 설정**을 알아야 합니다.

배우는 것
1. YOLO 탐지 데이터셋 폴더 구조
2. 라벨(.txt) 파일 형식 — 정규화 좌표
3. 데이터셋 YAML 작성법
4. `coco8` 실물로 구조 확인
5. 내 데이터셋을 만들 때 체크리스트

## 1. 탐지 데이터셋 폴더 구조

YOLO 탐지는 아래처럼 **이미지와 라벨을 짝지어** 둡니다.

```
my_dataset/
├── images/
│   ├── train/  img1.jpg, img2.jpg ...
│   └── val/    imgA.jpg ...
└── labels/
    ├── train/  img1.txt, img2.txt ...   (이미지와 같은 이름!)
    └── val/    imgA.txt ...
```

**핵심 규칙**: `images/train/img1.jpg` ↔ `labels/train/img1.txt` — 확장자만 다르고 이름이 같아야 짝이 맞습니다. 라벨이 없는(객체가 없는) 이미지는 빈 .txt 또는 파일 없음.

## 2. 라벨 파일(.txt) 형식

한 줄 = 객체 하나. 값은 **0~1로 정규화**된 좌표입니다 (이미지 크기로 나눈 값).

```
<class_id> <x_center> <y_center> <width> <height>
```

예: `0 0.512 0.383 0.244 0.706`
- class_id=0 (첫 번째 클래스)
- 박스 중심이 이미지 가로 51.2%, 세로 38.3% 지점
- 너비는 이미지 폭의 24.4%, 높이는 70.6%

> **왜 정규화?** 이미지 크기가 달라도 같은 라벨을 재사용할 수 있고, 학습 시 리사이즈에 영향받지 않기 때문.

## 3. 데이터셋 YAML

학습 시 `data=`에 넣는 파일. 데이터 위치와 클래스 이름을 알려줍니다.

```yaml
path: /abs/or/rel/my_dataset   # 데이터셋 루트
train: images/train            # path 기준 상대경로
val: images/val

names:                         # class_id -> 이름
  0: cat
  1: dog
```

`names`의 키(0,1,...)가 라벨 .txt의 `class_id`와 정확히 대응해야 합니다.

## 4. coco8 실물로 구조 확인하기

이전에 학습하면서 `datasets/coco8`이 자동 다운로드됐습니다. 실제 구조를 눈으로 봅시다.

In [ ]:
from ultralytics.utils import SETTINGS
from pathlib import Path

# Ultralytics가 데이터셋을 받는 기본 경로
root = Path(SETTINGS["datasets_dir"]) / "coco8"
print("coco8 위치:", root, "| 존재:", root.exists())

for p in sorted(root.rglob("*")):
    if p.is_dir():
        n = len(list(p.glob("*.*")))
        print(f"  {p.relative_to(root)}/  ({n} files)")

In [ ]:
# 라벨 파일 하나 열어보기
label_files = list((root / "labels").rglob("*.txt"))
sample = label_files[0]
print("라벨 파일:", sample.name)
print("-" * 40)
print(sample.read_text().strip())
print("-" * 40)
print("각 줄: class_id  x_center  y_center  width  height  (모두 0~1)")

In [ ]:
# coco8.yaml 내용 확인
import yaml
from ultralytics.utils import ROOT

yaml_path = ROOT / "cfg" / "datasets" / "coco8.yaml"
cfg = yaml.safe_load(yaml_path.read_text())
print("path :", cfg.get("path"))
print("train:", cfg.get("train"))
print("val  :", cfg.get("val"))
print("클래스 수:", len(cfg["names"]))
print("앞 5개 클래스:", {k: cfg["names"][k] for k in list(cfg["names"])[:5]})

## 5. 라벨을 이미지 위에 그려서 검증하기

정규화 좌표를 픽셀로 되돌려 박스를 그려보면, 라벨이 제대로 됐는지 눈으로 확인할 수 있습니다. (내 데이터 만들 때 필수 검증 습관)

In [ ]:
from PIL import Image, ImageDraw

img_files = list((root / "images").rglob("*.jpg"))
img_path = img_files[0]
lbl_path = Path(str(img_path).replace("/images/", "/labels/").replace(".jpg", ".txt"))

im = Image.open(img_path).convert("RGB")
W, H = im.size
draw = ImageDraw.Draw(im)

for line in lbl_path.read_text().strip().splitlines():
    cid, xc, yc, w, h = map(float, line.split())
    # 정규화 -> 픽셀, 중심좌표 -> 모서리좌표
    x1 = (xc - w / 2) * W; y1 = (yc - h / 2) * H
    x2 = (xc + w / 2) * W; y2 = (yc + h / 2) * H
    draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
    draw.text((x1, y1 - 10), str(int(cid)), fill="red")

print(f"{img_path.name}  ({W}x{H})")
im

## 6. 내 데이터셋 만들 때 체크리스트

- [ ] `images/` 와 `labels/` 를 train/val 로 나눴는가
- [ ] 이미지-라벨 파일명이 정확히 짝을 이루는가 (확장자만 다름)
- [ ] 라벨 좌표가 0~1 정규화 값인가
- [ ] `class_id`가 0부터 시작하고 YAML `names` 키와 일치하는가
- [ ] YAML의 `path/train/val` 경로가 맞는가
- [ ] 라벨을 이미지 위에 그려 눈으로 검증했는가

> **라벨링 도구 팁**: 직접 라벨링하려면 [Label Studio], [CVAT], [Roboflow] 등을 쓰고 'YOLO' 형식으로 내보내면 위 구조로 나옵니다.

**다음 노트북 →** `03_training_deep` : 학습 인자·하이퍼파라미터·로그 읽기